In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="talk")

The cell below defines a set of function for making the simulation.

You don't need to understand the details of these functions, so you can skip ahead to the next cell.

In [ ]:
def simulate_polygenic_trait_centered(n_loci, N=500, p=0.5, base_effect=1.0, env_sd=0.0):
    """
    Simulate a trait controlled by n_loci loci with additive effects.
    - Per-locus effect scaled so genetic variance comparable across n_loci.
    - Expected genetic mean subtracted so mean phenotype ≈ 0.
    """
    # HWE genotype probabilities
    probs = [(1 - p) ** 2, 2 * p * (1 - p), p**2]
    genotypes = np.random.choice([0, 1, 2], size=(N, n_loci), p=probs)

    # Scale per-locus effect
    effect_size = base_effect / np.sqrt(n_loci)

    # Raw genetic score and value
    genetic_score = genotypes.sum(axis=1)  # total number of "+" alleles
    genetic_value_raw = genetic_score * effect_size

    # Expected number of "+" alleles per individual
    expected_plus = 2 * p * n_loci
    expected_genetic_mean = expected_plus * effect_size

    # Center genetic value
    genetic_value = genetic_value_raw - expected_genetic_mean

    # Environmental noise
    env_noise = np.random.normal(loc=0.0, scale=env_sd, size=N)
    phenotype = genetic_value + env_noise

    return genotypes, genetic_score, phenotype


def plot_individuals_vs_trait(phenotypes, individual_ids, env_sds):
    """
    Create scatter plots of individuals vs trait values.

    Parameters:
    - phenotypes: list of 3 phenotype arrays
    - individual_ids: array of individual identifiers
    - env_sds: list of 3 environmental SDs for titles
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    colors = ["tab:blue", "tab:orange", "tab:green"]

    for i, (pheno, env_sd, color) in enumerate(zip(phenotypes, env_sds, colors)):
        axes[i].scatter(individual_ids, pheno, alpha=0.7, color=color)
        axes[i].set_title(f"{i + 1} gene(s) (env_sd = {env_sd})")
        axes[i].set_xlabel("Individual")
        axes[i].set_ylabel("Trait value" if i == 0 else "")

    plt.tight_layout()
    plt.show()


def plot_trait_distributions(phenotypes, env_sds, bins=15):
    """
    Create histograms of trait distributions.

    Parameters:
    - phenotypes: list of 3 phenotype arrays
    - env_sds: list of 3 environmental SDs for titles
    - bins: number of bins for histograms
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    colors = ["tab:blue", "tab:orange", "tab:green"]

    for i, (pheno, env_sd, color) in enumerate(zip(phenotypes, env_sds, colors)):
        axes[i].hist(pheno, bins=bins, alpha=0.8, color=color)
        axes[i].set_title(f"{i + 1} gene(s) – trait distribution")
        axes[i].set_xlabel("Trait value")
        axes[i].set_ylabel("Count" if i == 0 else "")

    plt.tight_layout()
    plt.show()


def plot_genotype_phenotype_scatter(genetic_scores, phenotypes):
    """
    Create scatter plots of genetic score vs phenotype.

    Parameters:
    - genetic_scores: list of 3 genetic score arrays
    - phenotypes: list of 3 phenotype arrays
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    colors = ["tab:blue", "tab:orange", "tab:green"]

    for i, (score, pheno, color) in enumerate(zip(genetic_scores, phenotypes, colors)):
        axes[i].scatter(score, pheno, alpha=0.7, color=color)
        axes[i].set_xticks(sorted(set(score)))
        axes[i].set_title(f"{i + 1} gene(s): genetic score vs trait")
        axes[i].set_xlabel("# of '+' alleles")
        axes[i].set_ylabel("Trait value" if i == 0 else "")

    plt.tight_layout()
    plt.show()


def print_summary_statistics(phenotypes):
    """
    Print summary statistics for phenotypes.

    Parameters:
    - phenotypes: list of 3 phenotype arrays
    """
    means = [np.mean(p) for p in phenotypes]
    sds = [np.std(p) for p in phenotypes]

    print("Means:", means)
    print("SDs:", sds)


def generate_trait_simulation_plots(N=500, p=0.5, env_sds=(0.0, 0.0, 1.0), base_effect=1.0, n_loci_list=(1, 2, 3)):
    """
    Main orchestration function: simulate polygenic traits and generate all plots.

    Parameters:
    - N: number of individuals per scenario
    - p: frequency of "+" (effect) allele at each locus
    - env_sds: tuple of environmental SDs for each scenario
    - base_effect: base genetic effect size
    - n_loci_list: tuple of number of loci for each scenario
    """
    # Run simulations
    simulations = []
    for n_loci, env_sd in zip(n_loci_list, env_sds):
        g, score, pheno = simulate_polygenic_trait_centered(
            n_loci=n_loci, N=N, p=p, base_effect=base_effect, env_sd=env_sd
        )
        simulations.append((g, score, pheno))

    # Extract components for easier handling
    genotypes, genetic_scores, phenotypes = zip(*simulations)
    individual_ids = np.arange(1, N + 1)

    # Generate all plots
    plot_individuals_vs_trait(phenotypes, individual_ids, env_sds)
    plot_trait_distributions(phenotypes, env_sds)
    plot_genotype_phenotype_scatter(genetic_scores, phenotypes)

    # Print summary
    print_summary_statistics(phenotypes)

The main function is `generate_trait_simulation_plots`, an example of how to use it is shown below.

In [ ]:
# Core parameters
N = 500
p = 0.5

# Environmental SD per gene category
env_sds = (0.0, 0.0, 1.0)

generate_trait_simulation_plots(N=N, p=p, env_sds=env_sds)